In [6]:
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import joblib
# ---------------- CONFIG ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_CLASSES = 7  # Modify according to your dataset

In [7]:
# ---------------- LOAD TRAINED MODEL ----------------
def build_model(num_classes):
    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )
    return model



In [8]:
model = build_model(NUM_CLASSES)
model.load_state_dict(torch.load("cnn_nour.pth", map_location=DEVICE))
model.eval().to(DEVICE)

# ---------------- FEATURE EXTRACTOR ----------------
feature_extractor = nn.Sequential(*list(model.children())[:-1])
feature_extractor.eval().to(DEVICE)

# ---------------- FEATURE EXTRACTION ----------------
def extract_features(dataloader, extractor):
    features, labels = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(dataloader, desc="Extracting features"):
            imgs = imgs.to(DEVICE)
            feats = extractor(imgs).squeeze()  # (B, 512, 1, 1) → (B, 512)
            if len(feats.shape) > 2:
                feats = feats.view(feats.size(0), -1)
            features.append(feats.cpu().numpy())
            labels.extend(lbls.numpy())
    return np.vstack(features), np.array(labels)



c:\Users\mohamed ahmed\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\mohamed ahmed\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\mohamed ahmed\AppData\Local\Temp\ipykernel_32020\1944448477.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbit

In [ ]:
# ---------------- LOAD DATALOADERS ----------------
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

DATA_DIR = "/kaggle/input/fer2013"
train_data = datasets.ImageFolder(f"./train", transform=transform)
val_data   = datasets.ImageFolder(f"./test", transform=transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

# ---------------- TRAIN LOGISTIC REGRESSION ----------------
X_train, y_train = extract_features(train_loader, feature_extractor)
X_val, y_val     = extract_features(val_loader, feature_extractor)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)


Extracting features:   8%|▊         | 36/449 [00:26<07:28,  1.09s/it]

In [ ]:
svm = SVC(kernel='linear')  # or try 'rbf' or 'poly'
svm.fit(X_train_scaled, y_train)
y_pred = svm.predict(X_val_scaled)
acc = accuracy_score(y_val, y_pred)
print(f"✅ SVM Accuracy: {acc:.4f}")
joblib.dump(svm, "svm_model.pkl")
joblib.dump(scaler, "feature_scaler.pkl")


In [ ]:
svm = joblib.load("svm_model.pkl")
scaler = joblib.load("feature_scaler.pkl")
